# Validação cruzada e ablação — TP1 E2

## Objetivo e protocolo

Comparação de textura, HOG e intensidade com SVR, Random Forest e Gradient Boosting em três dobras nos 2.800 IDs congelados de treino, com semente 42. O scaler e a baseline da média são ajustados exclusivamente no treino de cada dobra. A ablação compara textura em 224×224 e 128×128 pixels com Gradient Boosting.

O protocolo disponível é por imagem, sem chave verificável de paciente. A disjunção de IDs não comprova independência por paciente. Os conjuntos externos de validação e teste não são utilizados na seleção por esta execução.

## Execução no Google Colab

1. Utilizar uma sessão limpa e executar a célula de código.
2. No seletor de arquivos, carregar `atualizacao_fechamento_TP1.zip` e `matrizes_textura_RSNA_4000.zip`, juntos ou um por vez. A pasta local dos arquivos pode ser escolhida livremente. Sufixos como `(1)` e `(2)` são reconhecidos automaticamente.
3. A base é obtida pelo KaggleHub, com reutilização do cache quando disponível. Não é necessário carregar o ZIP das imagens manualmente. A extração e os treinamentos podem demorar.
4. Ao concluir, salvar `resultados_fechamento_TP1.zip` e uma cópia do notebook executado. O pacote contém métricas por dobra, previsões OOF, matrizes, versões e proveniência.

As porcentagens indicam marcos das etapas, não estimativas do tempo restante. Instalação e processamento usam subprocessos para reduzir a dependência de módulos previamente carregados. Esta rotina não publica alterações no GitHub. Os resultados existentes podem ser recuperados do pacote salvo sem repetir o processamento.


In [ ]:
from pathlib import Path
import csv
import hashlib
import io
import json
import re
import shutil
import subprocess
import sys
import tempfile
from zipfile import ZipFile
from google.colab import files

BASE = 'ed2ade6f303fc383f3248c12414cac564f09217f'
PROTOCOLO_PROVISORIO_POR_IMAGEM = True
print('0% — Envie os dois ZIPs. Pode selecionar um arquivo por vez.')
needed = ['atualizacao_fechamento_TP1.zip', 'matrizes_textura_RSNA_4000.zip']
uploaded = {}
while any(filename not in uploaded for filename in needed):
    missing = [filename for filename in needed if filename not in uploaded]
    print('Arquivo(s) ainda necessário(s): ' + ', '.join(missing))
    batch = files.upload()
    if not batch:
        raise RuntimeError('Seleção cancelada. Execute a célula novamente e envie os dois ZIPs, um por vez ou juntos.')
    for received_name, content in batch.items():
        canonical = re.sub(r"(?: \(\d+\))+(?=\.zip$)", "", Path(received_name).name)
        if canonical in needed:
            uploaded[canonical] = content
            print(f'Recebido: {received_name}')
        else:
            print(f'Arquivo não solicitado: {received_name}. Selecione um dos ZIPs indicados.')
print('Os dois ZIPs foram recebidos. Iniciando a preparação.')
work = Path(tempfile.mkdtemp(prefix='fechamento_tp1_', dir='/content'))
repo = work / 'repo'
subprocess.run(['git', 'clone', 'https://github.com/RaulFH0/tp1-e2-bone-age.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', BASE], check=True)
with ZipFile(io.BytesIO(uploaded[needed[0]])) as patch:
    manifest = json.loads(patch.read('PATCH_MANIFEST.json'))
    if manifest['base_commit'] != BASE:
        raise ValueError('Pacote de correção usa uma base diferente.')
    for filename, checks in manifest['files'].items():
        relative = Path(filename)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Caminho inválido no pacote.')
        destination = repo / relative
        expected = checks['base_sha256']
        if expected is None and destination.exists():
            raise ValueError(f'Arquivo novo já existe: {filename}')
        if expected is not None and (not destination.is_file() or
             hashlib.sha256(destination.read_bytes()).hexdigest() != expected):
            raise ValueError(f'Arquivo base diferente: {filename}')
        content = patch.read('repo/' + filename)
        if hashlib.sha256(content).hexdigest() != checks['new_sha256']:
            raise ValueError(f'Integridade inválida: {filename}')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements.txt'),
                'kagglehub==1.0.2'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=repo, check=True)
print('25% — Correções aplicadas em cópia isolada; testes aprovados.')
texture_root = work / 'texture'
texture_root.mkdir()
with ZipFile(io.BytesIO(uploaded[needed[1]])) as archive:
    for member in archive.infolist():
        relative = Path(member.filename)
        if relative.is_absolute() or '..' in relative.parts or (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Caminho inválido no pacote de textura.')
    archive.extractall(texture_root)
found = list(texture_root.rglob('texture_train.csv'))
if len(found) != 1:
    raise ValueError('Esperado exatamente um texture_train.csv no pacote.')
texture = found[0].parent
import kagglehub
handle = 'kmader/rsna-bone-age/versions/2'
dataset = Path(kagglehub.dataset_download(handle))
found = list(dataset.rglob('1377.png'))
if len(found) != 1:
    raise ValueError('Não foi possível localizar a pasta de PNGs sem ambiguidade.')
images = found[0].parent
if len(list(images.glob('*.png'))) != 12611:
    raise ValueError('Quantidade de PNGs diferente de 12.611.')
labels_found = list(dataset.rglob('boneage-training-dataset.csv'))
labels = (labels_found[0] if len(labels_found) == 1 else
          Path(kagglehub.dataset_download(handle, path='boneage-training-dataset.csv')))
with labels.open(newline='') as stream:
    rows = list(csv.DictReader(stream))
if len(rows) != 12611:
    raise ValueError('Quantidade de rótulos diferente de 12.611.')
print('50% — Imagens, rótulos e pacote disponíveis. Iniciando CV e ablação; esta etapa pode demorar.')
print('PROTOCOLO: por imagem; independência por paciente NÃO verificada. Esclarecimento com o professor continua pendente.')
command = [sys.executable, 'src/08_cross_validation.py', '--images-dir', str(images),
           '--csv', str(labels), '--texture-dir', str(texture), '--out-dir', str(work / 'results'),
           '--folds', '3', '--n-estimators', '300']
if PROTOCOLO_PROVISORIO_POR_IMAGEM:
    command.append('--allow-image-folds')
else:
    raise ValueError('Informe a chave verificável de paciente no comando antes de executar o modo agrupado.')
subprocess.run(command, cwd=repo, check=True)
result = json.loads((work / 'results/metrics_cross_validation.json').read_text())
if result['n_train'] != 2800 or result['n_folds'] != 3 or result['val_test_used_for_selection']:
    raise ValueError('Saída incompatível com o protocolo preparado.')
print('75% — Métricas reais e previsões OOF geradas. Conferindo o pacote.')
shutil.copy2(repo / 'docs/fechamento_execucao.md', work / 'results/LEIA_EXECUCAO.md')
shutil.copy2(repo / 'src/08_cross_validation.py', work / 'results/08_cross_validation.py')
shutil.copy2(repo / 'src/image_preprocessing.py', work / 'results/image_preprocessing.py')
(work / 'results/PATCH_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
archive_path = Path(shutil.make_archive(str(work / 'resultados_fechamento_TP1'), 'zip', work / 'results'))
with ZipFile(archive_path) as archive:
    if archive.testzip() is not None:
        raise ValueError('Falha de integridade no ZIP de saída.')
print('100% desta execução — resultados_fechamento_TP1.zip pronto.')
print('SHA-256:', hashlib.sha256(archive_path.read_bytes()).hexdigest())
print('Isso não encerra o TP1: artigo, agrupamento, avaliação final e contribuição assinada permanecem pendentes.')
files.download(str(archive_path))
